# Laboratorio 06 y 07 · Sesión 1
**Versión estudiante con TODO.** Curso CIIN1021P. Datos sintéticos para fines académicos.

Ejecuta las celdas **en orden**. En los bloques TODO, lee los comentarios numerados (empiezan con `#`) y reemplaza cada `____` por lo que indican. No ejecutes una celda con `____` pendientes.

Entorno: Colab CPU, PySpark 4.0.3, Java 17, Spark local[2].

**Duración: 90 minutos.** Preparación 15 · Ejercicio 01 30 · Ejercicio 02 35 · cierre y evidencias 10.

## Paso 0 · Preparar Colab
Sube este archivo en *Archivo > Subir cuaderno*. Usa un entorno CPU. Si se reinicia la máquina, vuelve a ejecutar este paso completo.

In [ ]:
# Paso 0. Instalar la versión utilizada en esta práctica.
%pip install -q pyspark==4.0.3

In [ ]:
# Paso 0. Verificar Java en la máquina de Colab.
import os, sys, subprocess, shutil
from pathlib import Path

java17 = Path('/usr/lib/jvm/java-17-openjdk-amd64')
if not java17.exists():
    # Esta instalación se realiza en la máquina temporal de Colab.
    subprocess.run(['apt-get', 'update', '-qq'], check=True)
    subprocess.run(['apt-get', 'install', '-y', '-qq',
                    'openjdk-17-jre-headless'], check=True)
os.environ['JAVA_HOME'] = str(java17)
os.environ['PATH'] = str(java17 / 'bin') + os.pathsep + os.environ['PATH']
os.environ['PYSPARK_PYTHON'] = sys.executable
print('Python:', sys.version.split()[0])
print(subprocess.check_output(['java', '-version'],
      stderr=subprocess.STDOUT, text=True))

In [ ]:
# Paso 0. Iniciar Spark Classic en una sola máquina con dos hilos.
from pyspark.sql import SparkSession, functions as F, types as T
spark = (SparkSession.builder.appName('UPN_Tema07')
         .master('local[2]')
         .config('spark.sql.shuffle.partitions', '2')
         .config('spark.sql.session.timeZone', 'UTC')
         .getOrCreate())
spark.sparkContext.setLogLevel('ERROR')
print('Spark:', spark.version)
print('Modo:', spark.sparkContext.master)
assert spark.range(5).count() == 5
print('Prueba de arranque correcta')

Spark: 4.0.3
Modo: local[2]
Prueba de arranque correcta


In [ ]:
# Paso 0. Subir únicamente datos_tema07.zip desde tu computadora.
from google.colab import files
from pathlib import Path
import zipfile

subidos = files.upload()
nombre = next((n for n in subidos if n.endswith('.zip')), None)
if nombre is None:
    raise ValueError('Sube el archivo datos_tema07.zip')
BASE = Path('/content/datos_tema07')
BASE.mkdir(exist_ok=True)
esperados = {'movitrack_control.csv', 'playas.csv', 'jornadas.csv',
             'residuos.csv', 'sensores.csv'}
with zipfile.ZipFile(nombre) as z:
    assert esperados.issubset(set(z.namelist())), 'ZIP de datos incorrecto'
    for archivo in esperados:
        (BASE / archivo).write_bytes(z.read(archivo))
print('Datos disponibles:', sorted(p.name for p in BASE.glob('*.csv')))

In [ ]:
# Paso 0. Función de ayuda para comprobar tus resultados.
# Compara lo que obtuviste con el valor esperado y muestra ✔ o ✘ (no necesitas entenderla).
ESPERADO = {
    'fecha_nulas': 6, 'total_viajes': 6, 'total_importe': 120.0,
    'por_servicio': {'ESTANDAR': (3, 60.0, 50.0), 'EJECUTIVO': (2, 35.0, 29.17),
                     'ENCOMIENDA': (1, 25.0, 20.83)},
    'promedio': {'Comas': (21.67, 30.0), 'Independencia': (25.0, 25.0), 'Los Olivos': (15.0, 20.0)},
    'particiones3': 3, 'particiones4': 4, 'comas_viajes': 3, 'comas_importe': 65,
    'totales_rdd': {'Comas': 65, 'Independencia': 25, 'Los Olivos': 30},
    'particiones_por_distrito': 2,
    'totales_csv': {'Comas': 65.0, 'Independencia': 25.0, 'Los Olivos': 30.0},
}
def verificar(nombre, obtenido):
    if obtenido == ESPERADO[nombre]:
        print('✔', nombre, 'correcto')
    else:
        print('✘', nombre, 'no coincide. Obtuviste:', obtenido)
        print('   Revisa los comentarios numerados del TODO y vuelve a ejecutar.')

## Referencia rápida (consúltala en los TODO)
Los comentarios de cada TODO te dicen **qué lograr**, no la palabra exacta. Elige de esta tabla, y si dudas, prueba en una celda aparte.

| Quiero… | Opciones en PySpark (hay distractores) |
|---|---|
| Preguntar por valores vacíos de una columna | `F.col('c').isNull()` · `F.col('c').isNotNull()` · `F.col('c').isin(...)` |
| Calcular por grupo (dentro de `agg`) | `F.count` · `F.sum` · `F.avg` · `F.min` · `F.max` · `F.round` |
| Saber cuántas particiones tiene un RDD | `rdd.getNumPartitions()` · `rdd.glom()` · `rdd.count()` · `rdd.take(n)` |
| Unir dos importes del mismo grupo | una operación entre `a` y `b` dentro de `lambda a, b: ...` |

*Si una opción no te convence, léela en voz alta: ¿qué pregunta responde?*

## Ejercicio 01 · Recuperar los controles del Tema 06 (30 min)

**Qué lograremos.** En el Tema 06 calculaste indicadores de MoviTrack con SQL Server y Power BI. Ahora comprobarás que Spark entrega **los mismos números**: 6 viajes y 120.00 de importe.

**Por qué importa.** Cambiar de herramienta no debe cambiar el significado de un indicador. Si los números difieren, el problema casi nunca es el motor: suele ser el tipo de dato, un filtro o la definición del indicador.

**Qué aprenderás:** leer un CSV con esquema explícito, detectar un esquema mal declarado, y calcular los mismos indicadores con SQL y con la API de DataFrames.

In [ ]:
# Paso 1. Leer el CSV de MoviTrack con un esquema explícito.
# El esquema le dice a Spark el nombre y el TIPO de cada columna.
# DECIMAL(12,2): hasta 12 dígitos con 2 decimales. Es exacto para dinero;
# un tipo decimal flotante (DOUBLE) puede acumular errores de redondeo.
esquema_viajes = ('CodigoViaje STRING, Fecha DATE, Servicio STRING, '
                  'Distrito STRING, Importe DECIMAL(12,2)')

viajes = (spark.read
          .option('header', True)       # la primera línea trae los nombres de columna
          .schema(esquema_viajes)       # usa NUESTRO esquema en lugar de adivinarlo
          .csv(str(BASE / 'movitrack_control.csv')))   # indica el archivo; todavía no lo lee

viajes.printSchema()   # muestra columnas y tipos
viajes.show()          # acción: ahora sí lee el archivo y muestra las filas

root
 |-- CodigoViaje: string (nullable = true)
 |-- Fecha: date (nullable = true)
 |-- Servicio: string (nullable = true)
 |-- Distrito: string (nullable = true)
 |-- Importe: decimal(12,2) (nullable = true)

+-----------+----------+----------+-------------+-------+
|CodigoViaje|     Fecha|  Servicio|     Distrito|Importe|
+-----------+----------+----------+-------------+-------+
|    CEN-001|2026-09-03| EJECUTIVO|        Comas|  15.00|
|    CEN-002|2026-09-04| EJECUTIVO|        Comas|  20.00|
|    CEN-003|2026-09-05|  ESTANDAR|   Los Olivos|  10.00|
|    CEN-004|2026-09-06|  ESTANDAR|   Los Olivos|  20.00|
|    CEN-005|2026-10-04|  ESTANDAR|        Comas|  30.00|
|    CEN-006|2026-10-05|ENCOMIENDA|Independencia|  25.00|
+-----------+----------+----------+-------------+-------+


In [ ]:
# TODO 1. ¿Qué pasa si el esquema no coincide con los datos?
# 1. Abajo se lee el MISMO archivo, pero declaramos Fecha como número entero (INT).
# 2. Ejecuta y observa la columna Fecha: Spark NO muestra ningún error.
# 3. Escribe la condición que seleccione las filas donde Fecha NO tiene valor.
#    Pista: en SQL sería WHERE Fecha IS ... ; busca el equivalente en la Referencia rápida.
# 4. Antes de ejecutar, anota cuántas filas esperas. Luego compara.
esquema_malo = ('CodigoViaje STRING, Fecha INT, Servicio STRING, '
                'Distrito STRING, Importe DECIMAL(12,2)')
viajes_malo = (spark.read.option('header', True).schema(esquema_malo)
               .csv(str(BASE / 'movitrack_control.csv')))
viajes_malo.show()

nulas = viajes_malo.filter(F.col('Fecha').isNull()).count()
print('Filas con Fecha nula:', nulas)
verificar('fecha_nulas', nulas)

+-----------+-----+----------+-------------+-------+
|CodigoViaje|Fecha|  Servicio|     Distrito|Importe|
+-----------+-----+----------+-------------+-------+
|    CEN-001| NULL| EJECUTIVO|        Comas|  15.00|
|    CEN-002| NULL| EJECUTIVO|        Comas|  20.00|
|    CEN-003| NULL|  ESTANDAR|   Los Olivos|  10.00|
|    CEN-004| NULL|  ESTANDAR|   Los Olivos|  20.00|
|    CEN-005| NULL|  ESTANDAR|        Comas|  30.00|
|    CEN-006| NULL|ENCOMIENDA|Independencia|  25.00|
+-----------+-----+----------+-------------+-------+

Filas con Fecha nula: 6
✔ fecha_nulas correcto


In [ ]:
# TODO 1b (reto). Reemplaza cada ____ por una línea de código completa escrita por ti.
# ¿Cuántas filas tienen el Importe nulo en viajes_malo? Antes de ejecutar, predice el resultado
# y explica en la Respuesta 1 por qué solo una columna quedó afectada.
# Reutiliza lo que acabas de aprender sobre filter y la condición de nulos.
nulas_importe = viajes_malo.filter(F.col('Importe').isNull()).count()
print('Filas con Importe nulo:', nulas_importe)

Filas con Importe nulo: 0


**Respuesta 1.** En dos líneas: ¿por qué es peligroso que Spark convierta en nulo un dato que no encaja en el tipo, sin avisar? ¿Por qué el Importe no quedó afectado? ¿Qué control del Paso 2 te habría hecho notar el problema?

**Respuesta modelo:** Spark no falla: convierte en NULL lo que no puede interpretar, y los indicadores salen mal sin ningún aviso. Contar filas y sumar importes (total esperado 6 viajes y 120.00) no detecta fechas nulas; sí lo habría hecho contar nulos por columna o revisar el esquema y el resultado de show().

In [ ]:
# Paso 2. Calcular los indicadores conocidos con SQL.
# createOrReplaceTempView registra el DataFrame como una "tabla" temporal
# llamada viajes, para poder consultarla con SELECT ... FROM viajes.
viajes.createOrReplaceTempView('viajes')

resumen_distrito = spark.sql("""
    SELECT Distrito,
           COUNT(*)     AS Viajes,
           SUM(Importe) AS ImporteTotal
    FROM viajes
    GROUP BY Distrito
    ORDER BY Distrito
""")
resumen_distrito.show()

# Controles generales. first() devuelve la primera fila; [0] toma su primer valor.
total_viajes = viajes.count()
total_importe = viajes.agg(F.sum('Importe')).first()[0]
print('Viajes:', total_viajes, '| Importe total:', total_importe)
verificar('total_viajes', total_viajes)
verificar('total_importe', float(total_importe))

+-------------+------+------------+
|     Distrito|Viajes|ImporteTotal|
+-------------+------+------------+
|        Comas|     3|       65.00|
|Independencia|     1|       25.00|
|   Los Olivos|     2|       30.00|
+-------------+------+------------+

Viajes: 6 | Importe total: 120.00
✔ total_viajes correcto
✔ total_importe correcto


In [ ]:
# TODO 2. Participación de cada servicio (el indicador "% por servicio" del Tema 06).
# 1. Escribe la columna Viajes: cuántos viajes hay en cada servicio.
# 2. Escribe la columna ImporteTotal: el dinero acumulado de cada servicio.
# 3. Escribe la columna por la que se agrupa: la que aparece en el SELECT sin función.
# 4. La columna Porcentaje ya está resuelta: SUM(SUM(Importe)) OVER () es el total
#    general de todos los servicios, y cada servicio se divide entre ese total.
#    (En DAX hacías esto con DIVIDE y ALL.)
por_servicio = spark.sql("""
    SELECT Servicio,
           COUNT(*)     AS Viajes,
           SUM(Importe) AS ImporteTotal,
           ROUND(100.0 * SUM(Importe) / SUM(SUM(Importe)) OVER (), 2) AS Porcentaje
    FROM viajes
    GROUP BY Servicio
    ORDER BY ImporteTotal DESC
""")
por_servicio.show()
verificar('por_servicio', {r.Servicio: (r.Viajes, float(r.ImporteTotal), float(r.Porcentaje))
                           for r in por_servicio.collect()})

+----------+------+------------+----------+
|  Servicio|Viajes|ImporteTotal|Porcentaje|
+----------+------+------------+----------+
|  ESTANDAR|     3|       60.00|     50.00|
| EJECUTIVO|     2|       35.00|     29.17|
|ENCOMIENDA|     1|       25.00|     20.83|
+----------+------+------------+----------+

✔ por_servicio correcto


In [ ]:
# TODO 3. Importe promedio por viaje en cada distrito, ahora con la API de DataFrames.
# 1. groupBy('Distrito') agrupa las filas, igual que GROUP BY en SQL.
# 2. agg(...) calcula varias métricas por grupo, separadas por coma.
#    Primera: el importe promedio por viaje, redondeado a dos decimales.
#    Segunda: el importe del viaje más caro de cada distrito.
# 3. Elige las dos funciones de la Referencia rápida (F.round ya está puesto).
promedio = (viajes.groupBy('Distrito')
            .agg(F.round(F.avg('Importe'), 2).alias('Promedio'),
                 F.max('Importe').alias('Maximo'))
            .orderBy('Distrito'))
promedio.show()
verificar('promedio', {r.Distrito: (float(r.Promedio), float(r.Maximo)) for r in promedio.collect()})

+-------------+--------+------+
|     Distrito|Promedio|Maximo|
+-------------+--------+------+
|        Comas|   21.67| 30.00|
|Independencia|   25.00| 25.00|
|   Los Olivos|   15.00| 20.00|
+-------------+--------+------+

✔ promedio correcto


**Equivalencias entre herramientas**

| Indicador | Tema 06 (SQL Server / Power BI) | Spark |
|---|---|---|
| Total de viajes | `COUNT(*)` | `viajes.count()` o `COUNT(*)` en `spark.sql` |
| Importe total | `SUM(Importe)` | `F.sum('Importe')` o `SUM(Importe)` |
| % por servicio | `DIVIDE` + `ALL` en DAX | `SUM(SUM(Importe)) OVER ()` |
| Promedio y máximo por distrito | `AVG(Importe)`, `MAX(Importe)` | `F.avg('Importe')`, `F.max('Importe')` |

**Respuesta 2.** Compara tus tablas con el dashboard del Tema 06. Si un total hubiera salido distinto, ¿qué tres cosas revisarías primero?

**Respuesta modelo:** 1) El esquema (tipos y fechas/importes nulos). 2) Los filtros aplicados (que no se hayan perdido filas). 3) La definición del indicador (qué se cuenta o suma y con qué grano). Un cambio de herramienta, por sí solo, no debería cambiar un total.

## Ejercicio 02 · Particiones, transformaciones y acciones (35 min)

**Qué lograremos.** Entender cómo Spark reparte los datos y cuándo realmente trabaja. Usaremos seis pares *(distrito, importe)* y después el mismo archivo CSV como texto.

**Ideas clave.**
- Una **partición** es una porción de los datos; cada tarea de Spark procesa una partición.
- Una **transformación** (`filter`, `map`, `reduceByKey`) solo *describe* el cálculo: no ejecuta nada.
- Una **acción** (`count`, `sum`, `collect`) pide un resultado y *dispara* el trabajo.

In [ ]:
# Paso 1. Crear un RDD con dos particiones a partir de seis pares (distrito, importe).
datos = [('Comas', 15), ('Comas', 20), ('Los Olivos', 10),
         ('Los Olivos', 20), ('Comas', 30), ('Independencia', 25)]

# parallelize reparte la lista en particiones; el 2 es el número de particiones.
rdd = spark.sparkContext.parallelize(datos, 2)
print('Particiones:', rdd.getNumPartitions())

# glom() agrupa los elementos de cada partición en una lista.
# collect() trae el resultado al driver: solo es seguro con muestras pequeñas.
print('Distribución:', rdd.glom().collect())

Particiones: 2
Distribución: [[('Comas', 15), ('Comas', 20), ('Los Olivos', 10)], [('Los Olivos', 20), ('Comas', 30), ('Independencia', 25)]]


**Predicción (antes de ejecutar la siguiente celda).** Si repartimos los mismos seis elementos en **3** particiones, ¿cuántos elementos tendrá cada una y cuáles? Y con **4**, ¿crees que el reparto será parejo?

**Respuesta modelo:** Con 3 particiones se espera 2 elementos por partición, en orden: [Comas 15, Comas 20], [Los Olivos 10, Los Olivos 20], [Comas 30, Independencia 25].

In [ ]:
# Paso 1b. Mismo RDD pero con 3 particiones: compara con tu predicción.
rdd3 = spark.sparkContext.parallelize(datos, 3)
print('Distribución con 3:', rdd3.glom().collect())
verificar('particiones3', rdd3.getNumPartitions())

# TODO 4. Ahora crea rdd4 con UNA partición más que rdd3.
# 1. Calcula tú el número (no lo escribe ningún comentario) y colócalo en el hueco.
# 2. Ejecuta: ¿los grupos quedaron del mismo tamaño? Responde en la Respuesta 3.
rdd4 = spark.sparkContext.parallelize(datos, 4)
print('Distribución con 4:', rdd4.glom().collect())
verificar('particiones4', rdd4.getNumPartitions())

Distribución con 3: [[('Comas', 15), ('Comas', 20)], [('Los Olivos', 10), ('Los Olivos', 20)], [('Comas', 30), ('Independencia', 25)]]
✔ particiones3 correcto
Distribución con 4: [[('Comas', 15)], [('Comas', 20), ('Los Olivos', 10)], [('Los Olivos', 20)], [('Comas', 30), ('Independencia', 25)]]
✔ particiones4 correcto


In [ ]:
# TODO 5. Tres transformaciones sobre el RDD original.
# Cada elemento de rdd es una tupla (distrito, importe):
# fila[0] es el distrito y fila[1] es el importe (se cuenta desde 0).
# 1. filter conserva solo los elementos para los que la función devuelve True.
#    Escribe la condición que sea verdadera cuando el distrito es Comas.
# 2. map transforma cada elemento. Haz que devuelva solo el importe de cada fila.
# 3. reduceByKey agrupa por la clave (el distrito) y va juntando de a DOS los importes
#    que comparten clave (a y b). Escribe cómo se combinan para obtener el total del distrito.
comas = rdd.filter(lambda fila: fila[0] == 'Comas')
importes_comas = comas.map(lambda fila: fila[1])
por_distrito = rdd.reduceByKey(lambda a, b: a + b)

# Hasta aquí NO se calculó nada: son transformaciones.
# Las siguientes líneas son ACCIONES: recién ahora Spark trabaja.
print('Viajes de Comas:', comas.count())
print('Importe de Comas:', importes_comas.sum())
print('Totales:', sorted(por_distrito.collect()))   # sorted: el orden de collect no está garantizado
verificar('comas_viajes', comas.count())
verificar('comas_importe', importes_comas.sum())
verificar('totales_rdd', dict(por_distrito.collect()))

Viajes de Comas: 3
Importe de Comas: 65
Totales: [('Comas', 65), ('Independencia', 25), ('Los Olivos', 30)]
✔ comas_viajes correcto
✔ comas_importe correcto
✔ totales_rdd correcto


In [ ]:
# TODO 6. Leer el plan de ejecución (linaje) de por_distrito.
# 1. toDebugString() muestra de qué RDD viene por_distrito, de abajo hacia arriba.
# 2. Ejecuta y busca la línea con "ShuffledRDD": ahí Spark REDISTRIBUYE los datos
#    entre particiones para juntar las claves iguales (algo parecido a un GROUP BY).
# 3. Completa el hueco con el método que dice cuántas particiones tiene el RDD (ver Referencia rápida).
print(por_distrito.toDebugString().decode())
n_part = por_distrito.getNumPartitions()
print('Particiones de por_distrito:', n_part)
verificar('particiones_por_distrito', n_part)

(2) PythonRDD[106] at collect at e2e:17 []
 |  MapPartitionsRDD[103] at mapPartitions at PythonRDD.scala:168 []
 |  ShuffledRDD[102] at partitionBy at NativeMethodAccessorImpl.java:0 []
 +-(2) PairwiseRDD[101] at reduceByKey at e2e:11 []
    |  PythonRDD[100] at reduceByKey at e2e:11 []
    |  ParallelCollectionRDD[94] at readRDDFromFile at PythonRDD.scala:297 []
Particiones de por_distrito: 2
✔ particiones_por_distrito correcto


In [ ]:
# Paso 2. El mismo cálculo, pero partiendo del archivo como TEXTO (sin esquema).
lineas = spark.sparkContext.textFile(str(BASE / 'movitrack_control.csv'))   # cada línea es un texto
print('Líneas:', lineas.count())         # 7 = 1 encabezado + 6 viajes

encabezado = lineas.first()              # la primera línea trae los nombres de columna
datos_csv = lineas.filter(lambda l: l != encabezado)   # descarta el encabezado
columnas = datos_csv.map(lambda l: l.split(','))       # separa cada línea por las comas
print(columnas.first())                  # observa que TODO es texto, incluso el importe

Líneas: 7
['CEN-001', '2026-09-03', 'EJECUTIVO', 'Comas', '15.00']


In [ ]:
# TODO 7. Totales por distrito desde el texto.
# 1. Mira la lista que imprimió columnas.first() en la celda anterior: ubica en qué
#    posición está el distrito y en cuál el importe. Se cuenta desde 0.
# 2. El importe llega como TEXTO ('15.00'): float(...) lo convierte a número antes de sumar.
# 3. Escribe las dos posiciones en los huecos.
totales_csv = (columnas.map(lambda c: (c[3], float(c[4])))
                       .reduceByKey(lambda a, b: a + b))
print(sorted(totales_csv.collect()))
verificar('totales_csv', dict(totales_csv.collect()))

[('Comas', 65.0), ('Independencia', 25.0), ('Los Olivos', 30.0)]
✔ totales_csv correcto


**Mismo resultado, tres caminos**

| | RDD desde texto | DataFrame con esquema | Spark SQL |
|---|---|---|---|
| Columnas | Por posición: `c[3]` | Por nombre: `'Distrito'` | Por nombre en SQL |
| Tipos | Todo es texto; hay que convertir | Declarados en el esquema | Los del DataFrame |
| Errores típicos | Posición equivocada, olvidar `float` | Esquema mal declarado → nulos | Vista no registrada |
| Conviene cuando | Datos sin estructura | Datos tabulares | Ya conoces SQL |

**Respuesta 3.** En cuatro líneas: 0) ¿por qué con 4 particiones el reparto no es parejo? 1) ¿qué instrucciones del Ejercicio 02 son acciones y cuáles transformaciones? 2) ¿por qué dos particiones no significan dos servidores? 3) Para un CSV de 20 columnas, ¿qué camino elegirías y por qué?

**Respuesta modelo:** 0) Seis elementos no se dividen en cuatro partes iguales: quedan grupos de 1 y de 2 elementos (Spark corta la lista en tramos consecutivos y no exige un reparto exacto). 1) Transformaciones: parallelize, filter, map, reduceByKey; acciones: count, sum, collect, first. 2) Una partición es una porción lógica de los datos; en local[2] las procesan dos hilos de la misma máquina, no dos servidores. 3) DataFrame (o SQL): se accede por nombre, los tipos están declarados y se evita contar posiciones y convertir a mano.

## Preguntas de cierre (responde en una línea cada una)
1. ¿Qué ventaja tiene declarar el esquema frente a dejar que Spark lo infiera?
2. ¿Qué indicador te habría avisado de que `Fecha` quedó mal leída?
3. Clasifica como transformación o acción: `filter`, `count`, `map`, `collect`, `reduceByKey`.
4. ¿Qué te indica la palabra `ShuffledRDD` en `toDebugString()`?
5. ¿Qué indicador de **tu desafío** podrías verificar con la misma idea de comparar herramientas?

*Mis respuestas:* ...

**Respuesta modelo:** 1) Los tipos quedan fijados y verificables (no depende de adivinanzas ni de una lectura extra del archivo), y se evita que Importe llegue como número flotante o texto. 2) Contar nulos por columna o mirar show(): Fecha aparecía como NULL en todas las filas. 3) Transformaciones: filter, map, reduceByKey; acciones: count, collect. 4) Que Spark redistribuye datos entre particiones para juntar las claves iguales (shuffle), como un GROUP BY. 5) Respuesta abierta: un KPI del desafío con su definición, y verificar que mantiene el valor al cambiar de herramienta.

## Cierre de la sesión 1
**Evidencia:** este cuaderno ejecutado de arriba hacia abajo y guardado como `.ipynb`, con las salidas visibles; captura de los ✔ de los TODO 1 a 7; y las respuestas 1, 2 y 3 escritas en las celdas.
La sesión 2 vuelve a cargar sus propios datos: no necesitas mantener esta sesión abierta.